In [12]:
!uv pip list | grep deepeval

Using Python 3.12.3 environment at: /home/vsc/LLM_TUNE/QA-FineTune/venv/venv-RAG
deepeval                           3.8.8


In [7]:
from deepeval.models.base_model import DeepEvalBaseLLM, DeepEvalBaseEmbeddingModel
from openai import OpenAI
from sentence_transformers import SentenceTransformer

class LocalExaone(DeepEvalBaseLLM):
    def __init__(self, model_name):
        self.model_name = model_name
        self.client = OpenAI(api_key="EMPTY", base_url="http://localhost:8002/v1")

    def load_model(self): return self.client

    def generate(self, prompt: str) -> str:
        res = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        return res.choices[0].message.content

    async def a_generate(self, prompt: str) -> str: return self.generate(prompt)
    def get_model_name(self): return self.model_name

# 2. Embedding 클래스 (get_model_name 필수!)
class LocalEmbedding(DeepEvalBaseEmbeddingModel):
    def __init__(self, model_name):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

    def load_model(self): return self.model
    def embed_text(self, text: str): return self.model.encode([text])[0].tolist()
    def embed_texts(self, texts: list[str]): return self.model.encode(texts).tolist()
    async def a_embed_text(self, text: str): return self.embed_text(text)
    async def a_embed_texts(self, texts: list[str]): return self.embed_texts(texts)
    def get_model_name(self): return self.model_name

In [9]:
from deepeval.models.base_model import DeepEvalBaseLLM, DeepEvalBaseEmbeddingModel
from deepeval.synthesizer import Synthesizer
from deepeval import set_active_embedding_model # 전역 설정 함수
from openai import OpenAI
from sentence_transformers import SentenceTransformer

local_llm = LocalExaone(model_name="/models/Exaone-3.5-32B-Instruct")
local_embed = LocalEmbedding(model_name="dragonkue/snowflake-arctic-embed-l-v2.0-ko")

# [핵심] 임베딩 모델을 전역적으로 활성화합니다.
# 이렇게 하면 Synthesizer가 내부적으로 이 모델을 찾아 씁니다.
set_active_embedding_model(local_embed)

# Synthesizer 초기화 (이제 embedder 파라미터를 빼도 에러가 안 납니다)
synthesizer = Synthesizer(model=local_llm)

print("준비 완료! 이제 데이터를 생성합니다.")

goldens = synthesizer.generate_goldens_from_docs(
    document_paths=['your_knowledge_base.pdf'],
    max_goldens_per_context=2,
    include_expected_output=True
)

goldens.save_as_json("exaone_dataset.json")

ImportError: cannot import name 'set_active_embedding_model' from 'deepeval' (/home/vsc/LLM_TUNE/QA-FineTune/venv/venv-RAG/lib/python3.12/site-packages/deepeval/__init__.py)